# Análise Exploratória de Dados

## Objetivo

Este conjunto de dados da IBM contém informações detalhadas sobre os funcionários da empresa. O objetivo é prever se um funcionário irá deixar a empresa (`Yes`) ou permanecer (`No`), caracterizando um problema de **classificação binária**. A variável alvo do modelo é `Attrition`.

## Importações

In [19]:
import pandas as pd
import numpy as np
import seaborn as sns

from scipy.stats import gaussian_kde

from plotly.subplots import make_subplots
import plotly.express as px
import plotly.graph_objects as go

from src.config import DADOS_ORIGINAIS, DADOS_TRATADOS

As constantes `DADOS_ORIGINAIS` e `DADOS_TRATADOS` são importadas de `src/config.py` e centralizam os caminhos dos arquivos de dados, facilitando a manutenção e evitando caminhos hardcoded ao longo do notebook.

## Análise inicial das colunas

In [2]:
df = pd.read_csv(DADOS_ORIGINAIS)

df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null   int64
 15

O dataset possui **1.470 registros e 35 colunas**, todas sem valores nulos. As colunas estão divididas entre tipos numéricos (`int64`) e textuais (`str`), sem necessidade de tratamento de missing values.

### Estatísticas Descritivas

In [4]:
with pd.option_context("float_format", "{:.2f}".format, "display.max_columns", 35):
    display(df.describe())

,Age,DailyRate,DistanceFromHome,Education,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,MonthlyIncome,MonthlyRate,NumCompaniesWorked,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
count,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00,1470.00
mean,36.92,802.49,9.19,2.91,1.00,1024.87,2.72,65.89,2.73,2.06,2.73,6502.93,14313.10,2.69,15.21,3.15,2.71,80.00,0.79,11.28,2.80,2.76,7.01,4.23,2.19,4.12
std,9.14,403.51,8.11,1.02,0.00,602.02,1.09,20.33,0.71,1.11,1.10,4707.96,7117.79,2.50,3.66,0.36,1.08,0.00,0.85,7.78,1.29,0.71,6.13,3.62,3.22,3.57
min,18.00,102.00,1.00,1.00,1.00,1.00,1.00,30.00,1.00,1.00,1.00,1009.00,2094.00,0.00,11.00,3.00,1.00,80.00,0.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00
25%,30.00,465.00,2.00,2.00,1.00,491.25,2.00,48.00,2.00,1.00,2.00,2911.00,8047.00,1.00,12.00,3.00,2.00,80.00,0.00,6.00,2.00,2.00,3.00,2.00,0.00,2.00
50%,36.00,802.00,7.00,3.00,1.00,1020.50,3.00,66.00,3.00,2.00,3.00,4919.00,14235.50,2.00,14.00,3.00,3.00,80.00,1.00,10.00,3.00,3.00,5.00,3.00,1.00,3.00
75%,43.00,1157.00,14.00,4.00,1.00,1555.75,4.00,83.75,3.00,3.00,4.00,8379.00,20461.50,4.00,18.00,3.00,4.00,80.00,1.00,15.00,3.00,3.00,9.00,7.00,3.00,7.00
max,60.00,1499.00,29.00,5.00,1.00,2068.00,4.00,100.00,4.00,5.00,4.00,19999.00,26999.00,9.00,25.00,4.00,4.00,80.00,3.00,40.00,6.00,4.00,40.00,18.00,15.00,17.00


A partir das estatísticas descritivas, identificamos três colunas com **desvio padrão igual a zero**, o que indica que todos os registros possuem o mesmo valor — portanto, não agregam informação ao modelo e podem ser removidas:

- `EmployeeCount`: constante igual a 1 para todos os registros.
- `StandardHours`: constante igual a 80 para todos os registros.

Além disso, a coluna `Education` armazena o nível educacional de cada funcionário como valor numérico ordinal (1 a 5), conforme descrito na documentação do dataset.

Outro ponto importante é a coluna `EmployeeNumber`. Pela sua natureza, ela parece ser um identificador único por funcionário. Vamos confirmar essa hipótese verificando a quantidade de valores distintos.

In [5]:
df['EmployeeNumber'].nunique()

1470

Confirmado: todos os 1.470 registros possuem um `EmployeeNumber` distinto, caracterizando-a como uma coluna de identificação. Por não carregar informação preditiva, ela também deve ser removida antes da modelagem.

In [6]:
df.describe(exclude='number')

,Attrition,BusinessTravel,Department,EducationField,Gender,JobRole,MaritalStatus,Over18,OverTime
count,1470,1470,1470,1470,1470,1470,1470,1470,1470
unique,2,3,3,6,2,9,3,1,2
top,No,Travel_Rarely,Research & Development,Life Sciences,Male,Sales Executive,Married,Y,No
freq,1233,1043,961,606,882,326,673,1470,1054


No `describe` das colunas não numéricas, é possível observar que a variável target apresenta desbalanceamento entre as classes, algo que precisará ser tratado posteriormente no notebook de machine learning. Também identificamos uma coluna com valor constante, `Over18`, que não agrega informação relevante ao modelo.


### Desbalanceamento

In [7]:
display(df['Attrition'].value_counts())
display(df['Attrition'].value_counts(normalize=True))

Attrition
No     1233
Yes     237
Name: count, dtype: int64

Attrition
No     0.838776
Yes    0.161224
Name: proportion, dtype: float64

Esse desbalanceamento, de aproximadamente 83% para 16%, precisa ser tratado na etapa de machine learning antes do treinamento dos modelos.

In [8]:
#todas as colunas com valores únicos

df.nunique()[df.nunique() == 1]

EmployeeCount    1
Over18           1
StandardHours    1
dtype: int64

### Duplicadas

In [9]:
df.duplicated().sum()

#não há linhas duplicadas

np.int64(0)

### Excluindo colunas 

In [10]:
df = df.drop(columns= df.nunique()[df.nunique() == 1.].index)
df = df.drop(columns=['EmployeeNumber'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 31 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EnvironmentSatisfaction   1470 non-null   int64
 9   Gender                    1470 non-null   str  
 10  HourlyRate                1470 non-null   int64
 11  JobInvolvement            1470 non-null   int64
 12  JobLevel                  1470 non-null   int64
 13  JobRole                   1470 non-null   str  
 14  JobSatisfaction           1470 non-null   int64
 15

## Tipos de colunas

Essa etapa tem como objetivo dar continuidade à análise exploratória e identificar os tipos de colunas, ou seja, verificar se elas são categóricas, numéricas, datas ou categorias representadas numericamente.

Para isso, é importante não apenas analisar a base de dados manualmente e entender os dados

In [11]:
colunas_categoricas_nao_ordenadas = [
    "BusinessTravel",
    "Department",
    "EducationField",
    "Gender",
    "JobRole",
    "MaritalStatus",
    "OverTime",
]

colunas_categoricas_ordenadas = [
    "Education",
    "EnvironmentSatisfaction",
    "JobSatisfaction",
    "JobInvolvement",
    "JobLevel",
    "PerformanceRating",
    "RelationshipSatisfaction",
    "StockOptionLevel",
    "WorkLifeBalance",
]

coluna_alvo = ["Attrition"]


colunas_numericas = [
    coluna for coluna in df.columns if coluna not in (
        colunas_categoricas_nao_ordenadas + colunas_categoricas_ordenadas + coluna_alvo
    )
]

## Análise gráfica

In [40]:
cores = px.colors.sequential.Blues

fig = make_subplots(
    rows=5,
    cols=4,
    subplot_titles=colunas_numericas
)

for i, coluna in enumerate(colunas_numericas):

    linha = i // 4 + 1
    coluna_pos = i % 4 + 1
    dados = df[coluna]
    fig.add_trace(
        go.Histogram(
            x=dados,
            histnorm='probability density',
            marker=dict(color=cores[i % len(cores)]),
        ),
        row=linha,
        col=coluna_pos
    )
    kde = gaussian_kde(dados)
    x_range = np.linspace(dados.min(), dados.max(), 200)

    fig.add_trace(
        go.Scatter(
            x=x_range,
            y=kde(x_range),
            mode='lines'
        ),
        row=linha,
        col=coluna_pos
    )

fig.update_layout(
    height=1000,
    width=1500,
    showlegend=False,
    template='plotly_dark',
     margin=dict(l=10, r=40, t=60, b=10)
)

fig.show()

A gente pode ver que `Distance from Home` tem uma assimetria, assim como outras colunas que requer tratamento, vamos confirmar isso nos boxplots

In [42]:
cores = px.colors.sequential.Blues

fig = make_subplots(
    rows=5,
    cols=4,
)

for i, coluna in enumerate(colunas_numericas):
    r = i // 4 + 1
    c = i % 4 + 1

    fig.add_trace(
        go.Box(
            x=df[coluna],
            name=coluna,
            quartilemethod="exclusive",
            showlegend=False,
            marker_color=cores[i % len(cores)]
        ),
        row=r,
        col=c
    )

    fig.update_xaxes(title_text=coluna, row=r, col=c)
    fig.update_yaxes(showticklabels=False, row=r, col=c)

fig.update_layout(
    template="plotly_dark",
    width=1300,
    height=900
)

fig.show()

É possível observar a presença de diversos outliers em algumas colunas, além de sinais de assimetria na distribuição dos dados. Essa assimetria pode ser identificada quando a caixa do boxplot aparece deslocada para um dos lados, indicando que os dados não estão distribuídos de forma simétrica. O que requer um preprocessamento

### boxplot por alvo do target

In [46]:
targets = df['Attrition'].unique()

color_map = {
    targets[0]: cores[2],
    targets[1]: cores[5]
}

fig = make_subplots(
    rows=5,
    cols=4,
)

for i, coluna in enumerate(colunas_numericas):
    r = i // 4 + 1
    c = i % 4 + 1

    for j, t in enumerate(targets):
        fig.add_trace(
            go.Box(
                x=df[df["Attrition"] == t][coluna],
                name=str(t),
                legendgroup=str(t),
                marker_color=color_map[t],
                showlegend=(i == 0)
            ),
            row=r,
            col=c
        )

    fig.update_xaxes(title_text=coluna, row=r, col=c, showgrid=False)
    fig.update_yaxes(title_text=None, row=r, col=c)

fig.update_layout(
    template="plotly_dark",
    width=1300,
    height=900
)

fig.show()

Como se trata de um problema de classificação, é importante analisar quais variáveis numéricas apresentam algum poder de discriminação em relação à variável alvo. A partir dos boxplots, observa-se que algumas colunas praticamente não contribuem para diferenciar as classes — como `HourlyRate`, cujas distribuições são muito semelhantes entre os grupos. Por outro lado, existem variáveis que aparentam fornecer algum nível de separação; entretanto, mesmo nesses casos, a distinção ainda não é claramente evidente apenas pela análise visual dos boxplots. Isso sugere que, isoladamente, essas variáveis podem ter capacidade limitada de discriminação, sendo necessário avaliá-las em conjunto com outras features durante o processo de modelagem.


### Correlação

In [47]:
corr = df.select_dtypes('number').corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
corr_masked = corr.mask(mask)

fig = px.imshow(corr_masked, aspect="auto", text_auto=".2f")
fig.update_xaxes(showgrid=False)
fig.update_yaxes(showgrid=False)
fig.update_layout(height=1500, template="plotly_dark")
fig.show()

É possível observar alguns padrões relevantes no gráfico de correlação. Um deles é a relação positiva entre o nível do cargo e o salário, o que indica que, conforme o nível hierárquico aumenta, a remuneração tende a crescer. Além disso, nota-se também uma correlação positiva entre a performance e o percentual de aumento salarial, sugerindo que colaboradores com melhor desempenho recebem, em média, aumentos salariais maiores, possivelmente refletindo algum tipo de política de bonificação ou recompensa baseada em performance.


## Exportação

In [48]:
df.to_parquet(DADOS_TRATADOS, index=False)